# Tutorial 07 — Uncertainty & Monte Carlo

Companion explainer: **07_uncertainty_monte_carlo.md**. Attach lognormal
distributions to exchanges, run Monte Carlo the bw2.5 way
(`use_distributions=True` + iteration), and — critically — compare two
alternatives under **shared** (dependent) sampling to get P(A > B).

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2 = find_flow("Carbon dioxide, fossil")
rng = np.random.default_rng(42)

## Build two kettle variants sharing a background (electricity)
A = coal grid (0.95), B = cleaner grid (0.55). Both use the SAME electricity
process, so their uncertainties are correlated.

In [2]:
DB = "t07_kettle"
if DB in bd.databases:
    del bd.databases[DB]
bd.Database(DB).write({
    (DB, "elec"): {"name": "electricity", "unit": "kilowatt hour", "exchanges": [
        {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
        {"input": co2.key, "amount": 0.95, "type": "biosphere"}]},
    (DB, "steel"): {"name": "steel", "unit": "kilogram", "exchanges": [
        {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 2.9, "type": "technosphere"},
        {"input": co2.key, "amount": 1.9, "type": "biosphere"}]},
    (DB, "A"): {"name": "kettle A (coal grid)", "unit": "unit", "exchanges": [
        {"input": (DB, "A"), "amount": 1.0, "type": "production"},
        {"input": (DB, "steel"), "amount": 1.2, "type": "technosphere"},
        {"input": (DB, "elec"), "amount": 2.0, "type": "technosphere"}]},
    (DB, "B"): {"name": "kettle B (light, more elec)", "unit": "unit", "exchanges": [
        {"input": (DB, "B"), "amount": 1.0, "type": "production"},
        {"input": (DB, "steel"), "amount": 0.9, "type": "technosphere"},
        {"input": (DB, "elec"), "amount": 3.2, "type": "technosphere"}]},
})

13:17:27-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

13:17:27-0400

 [

info     

] 

Vacuuming database            

## Attach lognormal uncertainty to every exchange
For lognormal (uncertainty type 2): `loc = ln(median)`, `scale = sigma` of the
underlying normal. Setting loc = amount (not ln!) is the classic blunder.

In [3]:
import math
for act in bd.Database(DB):
    for exc in act.exchanges():
        if exc["type"] == "production":
            continue
        amt = exc["amount"]
        exc["uncertainty type"] = 2          # lognormal
        exc["loc"] = math.log(amt)
        exc["scale"] = 0.15                  # ~ +-16% GSD
        exc.save()
bd.Database(DB).process()
A = bd.get_node(database=DB, code="A")
B = bd.get_node(database=DB, code="B")
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
det = {}
for lbl, act in [("A", A), ("B", B)]:
    l = bc.LCA({act: 1}, method=gwp); l.lci(); l.lcia(); det[lbl] = l.score
print("deterministic scores:", {k: round(v,3) for k,v in det.items()})

deterministic scores:

{'A': 7.486, 'B': 7.229}

## Monte Carlo on A alone

In [4]:
N = 500
mc = bc.LCA({A: 1}, method=gwp, use_distributions=True)
mc.lci(); mc.lcia()
scores_A = np.array([mc.score for _ in zip(range(N), mc)])
assert scores_A.std() > 0, "distributions not active!"
print(f"A: median={np.median(scores_A):.3f}, "
      f"90% CI=[{np.percentile(scores_A,5):.3f}, {np.percentile(scores_A,95):.3f}]")

A: median=7.618, 90% CI=[5.742, 10.108]

## Dependent comparison: sample once, evaluate BOTH demands per draw

In [5]:
cmp = bc.LCA({A: 1}, method=gwp, use_distributions=True)
cmp.lci(); cmp.lcia()
sa, sb = [], []
for _ in range(N):
    next(cmp)                      # resample matrices ONCE
    cmp.lcia(demand={A.id: 1}); sa.append(cmp.score)
    cmp.lcia(demand={B.id: 1}); sb.append(cmp.score)
sa, sb = np.array(sa), np.array(sb)
diff = sa - sb
p_A_worse = float(np.mean(diff > 0))
print(f"dependent: median A={np.median(sa):.3f}, median B={np.median(sb):.3f}")
print(f"P(A > B) under shared uncertainty = {p_A_worse:.2%}")

dependent: median A=7.392, median B=7.264

P(A > B) under shared uncertainty = 56.60%

## Independent comparison overstates overlap (the trap)

In [6]:
mcB = bc.LCA({B: 1}, method=gwp, use_distributions=True)
mcB.lci(); mcB.lcia()
scores_B_indep = np.array([mcB.score for _ in zip(range(N), mcB)])
# naive independent P(A>B): compare shuffled draws
p_indep = float(np.mean(scores_A > rng.permutation(scores_B_indep)))
print(f"P(A>B) dependent   = {p_A_worse:.2%}   (correct)")
print(f"P(A>B) independent = {p_indep:.2%}   (ignores shared background)")

P(A>B) dependent   = 56.60%   (correct)

P(A>B) independent = 53.20%   (ignores shared background)

## Plot: paired distributions + difference

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(sa, bins=30, alpha=0.6, label="A (coal grid)", color="#C44E52")
axes[0].hist(sb, bins=30, alpha=0.6, label="B (light)", color="#4C72B0")
axes[0].set_title("Paired MC score distributions"); axes[0].set_xlabel("kg CO2-eq")
axes[0].legend()
axes[1].hist(diff, bins=30, color="#8172B3")
axes[1].axvline(0, color="k", lw=1)
axes[1].set_title(f"A - B  (P(A>B)={p_A_worse:.0%})"); axes[1].set_xlabel("kg CO2-eq")
plt.tight_layout()
plt.savefig("tutorials_outputs_07.png", dpi=120, bbox_inches="tight")
print("saved tutorials_outputs_07.png")
plt.show()

saved tutorials_outputs_07.png

C:\Users\derne\AppData\Local\Temp\ipykernel_62592\4276379392.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Next: **08 — parameters & scenarios**.